# Customer Churn Prediction


# Bab 1. Exploratory Data Analysis (EDA)

## 1.1 Import Library

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

from model_utils import TARGET_COL, RANDOM_STATE, TEST_SIZE, engineer_customer_features, align_features, split_feature_types
from train_model import (
    build_direct_preprocessor,
    build_preprocessor,
    build_models,
    run_experiment,
    get_numeric_outlier_summary,
    aggregate_feature_importance,
    select_important_features,
    evaluate_classifier,
    metrics_to_row,
    build_metadata,
)

import joblib
import warnings
warnings.filterwarnings("ignore")

## 1.2 Load Dataset

In [ ]:
df = pd.read_csv("Sales - Marketing customer dataset.csv")

print("Jumlah baris dan kolom:", df.shape)
print("Nama kolom:")
print(df.columns.tolist())

## 1.3 Menampilkan 5 Baris Pertama

In [ ]:
df.head()

## 1.4 Informasi Dataset

In [ ]:
df.info()

## 1.5 Statistik Deskriptif

In [ ]:
df.describe(include="all")

## 1.6 Persentase Missing Value

In [ ]:
missing_percentage = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_percentage

## 1.7 Visualisasi Missing Value

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=missing_percentage.index, y=missing_percentage.values)
plt.xticks(rotation=90)
plt.xlabel("Kolom")
plt.ylabel("Persentase Missing Value (%)")
plt.title("Persentase Missing Value pada Setiap Kolom")
plt.tight_layout()
plt.show()

## 1.8 Distribusi Variabel Target Churn

In [ ]:
print(df[TARGET_COL].value_counts())
print(df[TARGET_COL].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COL)
plt.xlabel("Churn")
plt.ylabel("Jumlah Data")
plt.title("Distribusi Variabel Target Churn")
plt.show()

## 1.9 Heatmap Korelasi Fitur Numerik

In [ ]:
numeric_df = df.select_dtypes(include=["int64", "float64"])
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(16, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Heatmap Korelasi Fitur Numerik")
plt.tight_layout()
plt.show()

## 1.10 Korelasi Fitur Numerik terhadap Churn

In [ ]:
correlation_matrix[TARGET_COL].sort_values(ascending=False)

# Bab 2. Direct Modeling

Pada skenario direct, data digunakan sebagai baseline tanpa cleaning, outlier handling, feature selection, scaling, dan tuning. Adapter minimal tetap digunakan agar scikit-learn dapat membaca missing value dan fitur kategorikal.

## 2.1 Menentukan X dan y

In [ ]:
X_direct = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("Shape X:", X_direct.shape)
print("Shape y:", y.shape)

## 2.2 Train-Test Split

In [ ]:
X_train_direct, X_test_direct, y_train, y_test = train_test_split(
    X_direct,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Data latih:", X_train_direct.shape)
print("Data uji:", X_test_direct.shape)

## 2.3 Training 3 Model Direct

In [ ]:
direct_models = build_models(
    build_direct_preprocessor(X_train_direct),
    class_weight="balanced",
)

direct_rows, direct_fitted = run_experiment(
    "Direct Modeling",
    direct_models,
    X_train_direct,
    X_test_direct,
    y_train,
    y_test,
)

direct_results_df = pd.DataFrame(direct_rows)
direct_results_df

# Bab 3. Modeling Dengan Preprocessing

## 3.1 Pembersihan Data, Feature Engineering, dan Penghapusan Kolom Tidak Relevan

In [ ]:
duplicate_count = df.duplicated().sum()
print("Jumlah duplikasi:", duplicate_count)

df_model = engineer_customer_features(df)
df_model = df_model.drop_duplicates().reset_index(drop=True)

print("Shape setelah preprocessing awal:", df_model.shape)
df_model.head()

## 3.2 Menentukan X dan y Setelah Preprocessing

In [ ]:
X_clean = df_model.drop(columns=[TARGET_COL])
y_clean = df_model[TARGET_COL]

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean,
    y_clean,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_clean,
)

print("Data latih:", X_train_clean.shape)
print("Data uji:", X_test_clean.shape)

## 3.3 Ringkasan Outlier pada Data Latih

In [ ]:
outlier_summary_df = get_numeric_outlier_summary(X_train_clean)
outlier_summary_df

## 3.4 Training 3 Model Dengan Preprocessing

In [ ]:
preprocessing_models = build_models(
    build_preprocessor(X_train_clean),
    class_weight="balanced",
)

preprocessing_rows, preprocessing_fitted = run_experiment(
    "Preprocessing",
    preprocessing_models,
    X_train_clean,
    X_test_clean,
    y_train_clean,
    y_test_clean,
)

preprocessing_results_df = pd.DataFrame(preprocessing_rows)
preprocessing_results_df

# Bab 4. Hyperparameter Tuning dan Feature Selection

## 4.1 Feature Importance

In [ ]:
rf_importance_model = preprocessing_fitted["Random Forest"]["estimator"]
_, categorical_features_clean = split_feature_types(X_train_clean)

feature_importance_df = aggregate_feature_importance(
    rf_importance_model,
    categorical_features_clean,
)

feature_importance_df.head(20)

## 4.2 Visualisasi Feature Importance

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_df.head(15),
    x="importance",
    y="feature",
)
plt.title("Feature Importance Berdasarkan Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 4.3 Feature Selection

In [ ]:
selected_features = select_important_features(feature_importance_df)

X_train_selected = align_features(X_train_clean, selected_features)
X_test_selected = align_features(X_test_clean, selected_features)

numeric_selected, categorical_selected = split_feature_types(X_train_selected)

print("Jumlah fitur awal:", X_train_clean.shape[1])
print("Jumlah fitur terpilih:", len(selected_features))
print("Fitur terpilih:")
print(selected_features)

## 4.4 Grid Search Hyperparameter

In [ ]:
selected_preprocessor = build_preprocessor(X_train_selected)
grid_models = build_models(
    selected_preprocessor,
    class_weight="balanced",
)

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

parameter_grids = {}

for model_name, estimator in grid_models.items():
    estimator_step = estimator.steps[-1][0]
    estimator_object = estimator.steps[-1][1]

    if model_name == "Logistic Regression":
        parameter_grids[model_name] = [
            {
                f"{estimator_step}__C": [0.01, 0.1, 1, 10],
                f"{estimator_step}__penalty": ["l1", "l2"],
                f"{estimator_step}__solver": ["liblinear"],
                f"{estimator_step}__max_iter": [1000],
            }
        ]
    elif model_name == "Decision Tree":
        parameter_grids[model_name] = {
            f"{estimator_step}__criterion": ["gini", "entropy"],
            f"{estimator_step}__max_depth": [None, 5, 10, 20],
            f"{estimator_step}__min_samples_split": [2, 5, 10],
            f"{estimator_step}__min_samples_leaf": [1, 2, 4],
        }
    elif model_name == "Random Forest":
        parameter_grids[model_name] = {
            f"{estimator_step}__n_estimators": [100, 200],
            f"{estimator_step}__max_depth": [None, 10, 20],
            f"{estimator_step}__min_samples_split": [2, 5],
            f"{estimator_step}__min_samples_leaf": [1, 2],
            f"{estimator_step}__max_features": ["sqrt", "log2"],
        }
    elif model_name == "Voting Classifier":
        n_estimators = len(estimator_object.estimators)
        voting_weights = [None]

        for index in range(n_estimators):
            weight = [1] * n_estimators
            weight[index] = 2
            voting_weights.append(weight)

        parameter_grids[model_name] = {
            f"{estimator_step}__voting": ["hard", "soft"],
            f"{estimator_step}__weights": voting_weights,
        }

best_estimators = {}
grid_search_rows = []

for model_name, estimator in grid_models.items():
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=parameter_grids[model_name],
        scoring="f1",
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
    )

    grid_search.fit(X_train_selected, y_train_clean)

    best_estimators[model_name] = grid_search.best_estimator_
    grid_search_rows.append(
        {
            "model": model_name,
            "best_score": grid_search.best_score_,
            "best_params": grid_search.best_params_,
        }
    )

tuning_results_df = pd.DataFrame(grid_search_rows)
tuning_results_df = tuning_results_df.sort_values("best_score", ascending=False).reset_index(drop=True)
tuning_results_df


## 4.5 Evaluasi Best Estimator Hasil Grid Search

In [ ]:
tuning_rows = []
tuning_metrics_by_model = {}

for model_name, estimator in best_estimators.items():
    metrics = evaluate_classifier(estimator, X_test_selected, y_test_clean)
    tuning_rows.append(metrics_to_row("Grid Search + Feature Selection", model_name, metrics))
    tuning_metrics_by_model[model_name] = metrics

    print("=" * 60)
    print(model_name)
    print("Accuracy :", metrics["accuracy"])
    print("Precision:", metrics["precision"])
    print("Recall   :", metrics["recall"])
    print("F1-score :", metrics["f1_score"])
    print("Confusion Matrix:")
    print(np.asarray(metrics["confusion_matrix"]))
    print()

tuned_results_df = pd.DataFrame(tuning_rows)
tuned_results_df

## 4.6 Ringkasan Total 9 Model

In [ ]:
all_results_df = pd.concat(
    [direct_results_df, preprocessing_results_df, tuned_results_df],
    ignore_index=True,
)

all_results_df.sort_values("f1_score", ascending=False)

# Bab 5. Deployment ke Streamlit Cloud

## 5.1 Simpan Model Terbaik dan Metadata

In [ ]:
best_row = all_results_df.sort_values("f1_score", ascending=False).iloc[0]
best_model_name = best_row["model"]
best_scenario = best_row["scenario"]

if best_scenario == "Grid Search + Feature Selection":
    best_model = best_estimators[best_model_name]
    best_metrics = tuning_metrics_by_model[best_model_name]
    best_params = tuning_results_df.loc[tuning_results_df["model"] == best_model_name, "best_params"].iloc[0]
    final_features = selected_features
    final_numeric = numeric_selected
    final_categorical = categorical_selected
else:
    raise ValueError("Model terbaik sebaiknya berasal dari skenario grid search untuk deployment final.")

reference_date = pd.to_datetime(df["last_purchase_date"], errors="coerce").max()
metadata = build_metadata(
    df_raw=df,
    df_model=df_model,
    selected_features=final_features,
    numeric_features=final_numeric,
    categorical_features=final_categorical,
    best_model_name=best_model_name,
    best_metrics=best_metrics,
    best_params=best_params,
    results_df=all_results_df,
    feature_importance_df=feature_importance_df,
    reference_date=reference_date,
)
metadata["best_scenario"] = best_scenario

joblib.dump(best_model, "best_model.joblib")
joblib.dump(metadata, "model_metadata.joblib")

print("Model terbaik:", best_model_name)
print("Skenario:", best_scenario)
print("File best_model.joblib dan model_metadata.joblib berhasil dibuat.")